# Prestige vs. Performance: Visualizing Biases in Global University Rankings

**Course:** Data Visualisation [B-KUL-G0R04a] (KU Leuven)  
**Target Audience:** Prospective International Master's & PhD Students  
**Team Members:** YIN Renlong, Victor Kao, Lei Pei, Szabó Gergely, Kawtar Darkaoui, Deborah Adelakun  
**Date:** 27 April 2026  

---

## 1. Project Motivation & Narrative
Millions of prospective students rely on global university rankings to make life-altering educational and financial decisions. However, these ranking systems often compress highly diverse institutions into a single hierarchical scale, masking severe methodological biases. 

This project applies **visual analytics and interaction techniques** to deconstruct and compare two major ranking bodies: the **Times Higher Education (THE)** rankings (which heavily weigh measurable research and industry output) against the **QS World University Rankings** (which lean heavily on accumulated academic and employer "reputation"). 

By visualizing this data, we aim to guide our target audience—prospective international students—to see beyond historical brand prestige. Our interactive visualizations are designed to uncover the "Hidden Gems": specialized or rising universities (particularly in Asia and Europe) that offer world-class research infrastructure but are undervalued by global reputation metrics.

## 2. Visualization & Interaction Objectives
Aligned with the core principles of data visualization, this interactive notebook utilizes **Plotly** to implement the following design choices:
*   **Effective Visual Encodings:** We utilize spatial position along common scales (scatter plots) to ensure the highest accuracy in quantitative graphical perception (Cleveland & McGill, 1984) when comparing competing ranking scores.
*   **Interaction Techniques:** To manage the high dimensionality and visual clutter of over 1,000 global universities without violating Gestalt principles, we implement interactive tooltips (details-on-demand) and legend-filtering to support deeper exploratory analysis (Heer & Shneiderman, 2012).
*   **Data Augmentation:** We integrate live socio-economic data (GDP) to provide a broader visual context on how national wealth drives academic standing (Pietrucha, 2018).

## 3. Data Sources
1.  **Primary Dataset:** [THE World University Rankings 2016-2026](https://www.kaggle.com/datasets/raymondtoo/the-world-university-rankings-2016-2024) (Longitudinal research metrics).
2.  **Comparison Dataset:** [2026 QS World University Rankings](https://www.kaggle.com/datasets/akashbommidi/2026-qs-world-university-rankings) (Reputation metrics).
3.  **Augmentation Source:** **World Bank Open Data API** (Live GDP per capita, current US$).

## 4. Bibliography
* **Bowman, N. A., & Bastedo, M. N. (2011).** Anchoring effects in world university rankings: Exploring biases in reputation scores. *Higher Education*, 61(4), 431–444.
* **Cleveland, W. S., & McGill, R. (1984).** Graphical perception: Theory, experimentation, and application to the development of graphical methods. *Journal of the American Statistical Association*, 79(387), 531–554.
* **Heer, J., & Shneiderman, B. (2012).** Interactive dynamics for visual analysis. *Communications of the ACM*, 55(4), 45–54.
* **Pietrucha, J. (2018).** Country-specific determinants of world university rankings. *Scientometrics*, 114(3), 1129–1139.
* **Shin, J. C., Toutkoushian, R. K., & Teichler, U. (Eds.). (2011).** *University rankings: Theoretical basis, methodology and impacts on global higher education*. Springer.
* **Yi, J. S., Kang, Y.-A., Stasko, J. T., & Jacko, J. A. (2007).** Toward a deeper understanding of the role of interaction in information visualization. *IEEE Transactions on Visualization and Computer Graphics*, 13(6), 1224–1231.

---

In [1]:
# Install necessary libraries
# %pip install kagglehub pandas plotly thefuzz python-Levenshtein ipywidgets

# Import necessary modules
import warnings
warnings.filterwarnings('ignore')  # <--- THIS HIDES THE ANNOYING RED WARNINGS!

import pandas as pd
import plotly.express as px  
import plotly.graph_objects as go 
import plotly.io as pio          
import re 
import kagglehub 
import os 
from scipy.stats import zscore 
from thefuzz import process 

# --- CRITICAL FIX FOR WEB EXPORT ---
# This forces Plotly to generate interactive Javascript instead of flat PNG fallbacks
pio.renderers.default = 'notebook_connected'

print("Libraries imported successfully. Ready for interactive plotting!")

Libraries imported successfully. Ready for interactive plotting!


In [2]:
# Load the primary dataset directly from Kaggle, and download latest version
path_times_dir = kagglehub.dataset_download("raymondtoo/the-world-university-rankings-2016-2024")

print("Path to dataset files:", path_times_dir)

# Automatically find the CSV file in the downloaded directory (handles the file regardless of the exact filename)
csv_files_times = [f for f in os.listdir(path_times_dir) if f.endswith('.csv')]
file_path_times = os.path.join(path_times_dir, csv_files_times[0])

df_times = pd.read_csv(file_path_times)

# --- Data Cleaning ---

# 1. Clean 'Student Population': Remove commas (e.g., "22,005" to 22005)
df_times['Student Population'] = df_times['Student Population'].astype(str).str.replace(',', '', regex=True)
df_times['Student Population'] = pd.to_numeric(df_times['Student Population'], errors='coerce')

# 2. Clean 'International Students': Remove percentage sign (e.g., "26%" to 26)
df_times['International Students'] = df_times['International Students'].astype(str).str.replace('%', '', regex=True)
df_times['International Students'] = pd.to_numeric(df_times['International Students'], errors='coerce')

# 3. Ensure Rank is integer
df_times['Rank'] = pd.to_numeric(df_times['Rank'], errors='coerce').fillna(999).astype(int)

# 4. Standardize Column Names (Best Practice)
df_times.columns = [col.lower().replace(' ', '_') for col in df_times.columns]

print("Data Cleaned.")
print(df_times.info())
df_times.sample(10) # Shows 10 random universities for verifying

Path to dataset files: /Users/Renlong/.cache/kagglehub/datasets/raymondtoo/the-world-university-rankings-2016-2024/versions/5
Data Cleaned.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16713 entries, 0 to 16712
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   rank                     16713 non-null  int64  
 1   name                     16713 non-null  object 
 2   country                  16713 non-null  object 
 3   student_population       16713 non-null  float64
 4   students_to_staff_ratio  16713 non-null  float64
 5   international_students   16703 non-null  float64
 6   female_to_male_ratio     15953 non-null  object 
 7   overall_score            16713 non-null  float64
 8   teaching                 16713 non-null  float64
 9   research_environment     16713 non-null  float64
 10  research_quality         16713 non-null  float64
 11  industry_impact          16713 non-null  flo

,rank,name,country,student_population,students_to_staff_ratio,international_students,female_to_male_ratio,overall_score,teaching,research_environment,research_quality,industry_impact,international_outlook,year
16158,1637,University of Zimbabwe,Zimbabwe,24488.0,19.1,1.0,58 : 42,25.0865,14.2,10.6,43.8,20.9,51.3,2026
11909,1384,Universitas Syiah Kuala,Indonesia,30701.0,20.4,0.0,62:38:00,25.0210,14.0,9.4,51.5,19.5,25.8,2024
2523,743,Czech Technical University in Prague,Czech Republic,21266.0,13.2,16.0,30 : 70,24.1800,20.2,17.3,28.5,39.3,45.3,2018
5241,1100,London South Bank University,United Kingdom,12170.0,16.6,33.0,56:44:00,19.6875,15.4,10.4,18.1,34.8,75.3,2020
8639,1575,Iwate University,Japan,5382.0,13.6,3.0,38 : 62,14.7075,20.2,11.3,8.3,37.2,24.5,2022
6489,951,Normandy University,France,67205.0,20.4,11.0,55:45:00,26.2550,19.9,14.9,36.4,34.4,53.8,2021
14778,257,Vita-Salute San Raffaele University,Italy,4573.0,27.6,6.0,64 : 36,56.1180,34.3,34.0,98.8,66.3,51.3,2026
11082,557,Polytechnic University of Bari,Italy,10460.0,43.0,1.0,29 : 71,43.4425,23.0,27.6,76.9,59.9,42.5,2024
14795,274,Northwestern Polytechnical University,China,35982.0,13.6,5.0,29 : 71,55.5290,41.0,42.4,78.6,99.7,47.6,2026
4582,441,Rensselaer Polytechnic Institute,United States,7798.0,15.6,20.0,32 : 68,40.8625,30.0,28.7,58.9,67.0,52.1,2020


In [3]:
# --- Define Regions ---
anglosphere = ['United States', 'United Kingdom', 'Australia', 'Canada', 'New Zealand', 'Ireland']
asian_tigers = ['China', 'Japan', 'South Korea', 'Singapore', 'Hong Kong']
eu_majors = ['Germany', 'France', 'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 'Italy', 'Spain']

def get_region(country):
    if country in anglosphere:
        return 'Anglosphere (US/UK/etc)'
    elif country in asian_tigers:
        return 'Asia (Major Economies)'
    elif country in eu_majors:
        return 'European Union (Major)'
    else:
        return 'Rest of World'

df_times['region_group'] = df_times['country'].apply(get_region)

# Filter for Top 200
df_top200 = df_times[df_times['rank'] <= 200].copy()
region_trends = df_top200.groupby(['year', 'region_group']).size().unstack().fillna(0).reset_index()

# --- INTERACTIVE VISUALIZATION ---
fig = px.line(
    region_trends, 
    x='year', 
    y=region_trends.columns[1:], 
    title='Number of Universities in Top 200 by Region (2016-2026)',
    labels={'value': 'Number of Universities', 'year': 'Year', 'variable': 'Region'},
    markers=True,
    template='plotly_white'
)
fig.show()

# --- GRAPH SUMMARY ---
print("\n--- GRAPH SUMMARY ---")
region_summary = df_top200.groupby(['year', 'region_group']).size().unstack()
print("University Counts in Top 200 (2016 vs 2026):")
print(region_summary.loc[[2016, 2026]].transpose().to_string())


--- GRAPH SUMMARY ---
University Counts in Top 200 (2016 vs 2026):
year                     2016  2026
region_group                       
Anglosphere (US/UK/etc)   115   101
Asia (Major Economies)     13    32
European Union (Major)     60    54
Rest of World              12    13


**Observation:**
The longitudinal analysis (2016-2026) reveals a distinct geopolitical shift in the global academic elite (Top 200):

*   **Anglosphere (Decline):** While English-speaking countries still hold the majority of spots, their dominance is eroding. The count dropped from approximately **115 universities in 2016 to around 100 in 2026**.
*   **Asia (Surge):** The "Asian Tigers" show the most aggressive growth, **more than doubling** their presence in the Top 200, rising from ~13 institutions to over 30 in the span of a decade.
*   **European Union (Slight Erosion):** The EU shows a slow, gradual decline, dropping from ~60 to ~54 institutions, suggesting it is struggling to keep pace with the rapid rise of Asian research powerhouses.

In [4]:
# 1. Load the QS dataset directly from Kaggle
path_qs_dir = kagglehub.dataset_download("akashbommidi/2026-qs-world-university-rankings")

# Automatically find the CSV file
csv_files_qs = [f for f in os.listdir(path_qs_dir) if f.endswith('.csv')]
file_path_qs = os.path.join(path_qs_dir, csv_files_qs[0])
df_qs = pd.read_csv(file_path_qs)

# 2. Clean QS Ranks to be Numeric
def convert_rank(rank):
    rank = str(rank)
    rank = rank.replace('=', '')         # Remove '=' (e.g., "10=" to "10")
    if '-' in rank:
        rank = rank.split('-')[0]        # Handle ranges (e.g., "601-650" to "601")
    rank = rank.replace('+', '')         # Handle plus signs (e.g., "601+" to "601")
    return rank

df_qs['2026 Rank'] = df_qs['2026 Rank'].apply(convert_rank)
df_qs['2026 Rank'] = pd.to_numeric(df_qs['2026 Rank'], errors='coerce')

# 3. Filter Times data for 2026 only
# Note: This assumes 'df_times' was loaded in a previous cell. 
df_times_2026 = df_times[df_times['year'] == 2026].copy()

# 4. Name Cleaning (Robust)
def clean_university_name(name):
    name = str(name).lower()
    # Remove text inside parentheses (e.g., "(MIT)" or "(KCL)")
    name = re.sub(r'\s*\(.*?\)', '', name)
    # Replace "smart quotes" (’) with straight quotes (')
    name = name.replace('’', "'")
    # Remove "the " from the start
    name = re.sub(r'^the\s+', '', name)
    return name.strip()

# Apply cleaning
df_times_2026['name_clean'] = df_times_2026['name'].apply(clean_university_name)
df_qs['name_clean'] = df_qs['Institution Name'].apply(clean_university_name)

# 5. FUZZY MATCHING LOGIC
# Get unique lists of names
times_names = df_times_2026['name_clean'].unique()
qs_names = df_qs['name_clean'].unique()
name_mapping = {}

print("Running fuzzy matching (checks all universities, maybe a little bit slow...)")

for qs_name in qs_names:
    # A. Try exact match first (Fast)
    if qs_name in times_names:
        name_mapping[qs_name] = qs_name
    # B. If no exact match, try Fuzzy Match (Slower, but smarter.....)
    else:
        # Returns tuple: (BestMatchName, Score)
        match_tuple = process.extractOne(qs_name, times_names)
        if match_tuple:
            best_match_name = match_tuple[0]
            score = match_tuple[1]
            # Threshold: Only accept if 90% similar or higher
            if score >= 90:
                name_mapping[qs_name] = best_match_name

# Apply the mapping to QS data
df_qs['merge_name'] = df_qs['name_clean'].map(name_mapping)

# Drop rows where it couldn't find a match
df_qs_matched = df_qs.dropna(subset=['merge_name'])

# 6. Merge using the aligned names
df_merged = pd.merge(
    df_times_2026,
    df_qs_matched,
    left_on='name_clean',
    right_on='merge_name',
    how='inner',
    suffixes=('_times', '_qs')
)

print(f"Successfully matched {len(df_merged)} universities between Times and QS.")

# PREPARE FOR PLOTTING (Add Regions)

# Define Regions (Redefined here to ensure they exist)
anglosphere = ['United States', 'United Kingdom', 'Australia', 'Canada', 'New Zealand', 'Ireland']
asian_tigers = ['China', 'Japan', 'South Korea', 'Singapore', 'Hong Kong']
eu_majors = ['Germany', 'France', 'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 'Italy', 'Spain']

def get_region(country):
    if country in anglosphere:
        return 'Anglosphere (US/UK/etc)'
    elif country in asian_tigers:
        return 'Asia (Major Economies)'
    elif country in eu_majors:
        return 'EU (Major)'
    else:
        return 'Rest of World'

# Apply region logic to the NEW merged dataset
df_merged['region_group'] = df_merged['country'].apply(get_region)

# 7. INTERACTIVE Visualization: Correlation with REGION COLORS
fig = px.scatter(
    df_merged, 
    x='rank', 
    y='2026 Rank', 
    color='region_group',
    hover_name='name_clean_times', # Shows University Name on hover!
    hover_data=['country'],
    title='Comparison of Rankings 2026: Times vs. QS',
    labels={'rank': 'Times Higher Education Rank', '2026 Rank': 'QS World Rank'},
    template='plotly_white',
    width=900,
    height=700
)

# Add a diagonal line for reference (x=y)
fig.add_shape(type="line", x0=0, y0=0, x1=250, y1=250, line=dict(color="red", dash="dash"))

fig.update_xaxes(range=[0, 250])
fig.update_yaxes(range=[0, 250])
fig.show()

# --- GRAPH SUMMARY ---
print("\n--- GRAPH SUMMARY ---")
rank_corr = df_merged['rank'].corr(df_merged['2026 Rank'])
print(f"Pearson Correlation between Times and QS Ranks: {rank_corr:.2f}")
print(f"Number of universities in common: {len(df_merged)}")

Running fuzzy matching (checks all universities, maybe a little bit slow...)
Successfully matched 1117 universities between Times and QS.



--- GRAPH SUMMARY ---
Pearson Correlation between Times and QS Ranks: 0.77
Number of universities in common: 1117


### Observation: Agreement between Ranking Systems
The scatter plot above compares the *Times Higher Education (2026)* rank against the *QS World University Rankings (2026)*.

* **General Trend:** There is a clear positive trend (Correlation: 0.77), though the strength of agreement is slightly lower than in smaller samples, reflecting the diversity of this expanded dataset.
* The color coding reveals a distinct "Anglosphere Consensus." Universities in the US, UK, and Australia (Blue dots) cluster tightly along the diagonal, indicating that both systems agree on their standing.
* In contrast, Asian and "Rest of World" institutions show significantly more scatter. This suggests that the choice of ranking system (reputation-heavy QS vs. citation-heavy Times) has a much larger impact on the perceived prestige of non-Western universities.

In [5]:
# Analysis: Tracking the Elite (Dynamic Selection)

# 1: Find the "Elite" Universities
# Ask Python "Give me the names of any university 
# that has been ranked in the Top 10 at ANY point between 2016 and 2026."
elite_filter = df_times['rank'] <= 10
elite_universities_list = df_times[elite_filter]['name'].unique()

# 2: Filter the data
df_elite_trajectory = df_times[df_times['name'].isin(elite_universities_list)].copy()

# 3: INTERACTIVE Visualization with Plotly (Replaces Seaborn/Matplotlib)
fig = px.line(
    df_elite_trajectory, 
    x='year', 
    y='rank', 
    color='name',        # Automatically assigns distinct colors!
    markers=True,
    hover_name='name',   # Shows the university name when you hover!
    title='Stability of the Elite Universities: Trajectories (2016-2026)',
    labels={'rank': 'World Rank', 'year': 'Year', 'name': 'University'},
    template='plotly_white',
    width=1000,
    height=600
)

# 4: Formatting
# Invert Y-axis so Rank 1 goes at the top!
fig.update_yaxes(autorange="reversed", tickmode='linear', tick0=1, dtick=1)
# Restrict X-axis slightly so it looks neat
fig.update_xaxes(range=[2015.8, 2026.2])

fig.show()

# --- GRAPH SUMMARY ---
print("\n--- GRAPH SUMMARY ---")
# Show the Top 5 status in the final year
top_5_2026 = df_times[(df_times['year'] == 2026) & (df_times['rank'] <= 5)][['rank', 'name']]
print("The Top 5 Universities in 2026:\n", top_5_2026.to_string(index=False))


--- GRAPH SUMMARY ---
The Top 5 Universities in 2026:
  rank                                  name
    1                  University of Oxford
    2 Massachusetts Institute of Technology
    3                  Princeton University
    4               University of Cambridge
    5                   Stanford University



### Analysis of Elite Stability (2016-2026)

**Visual Observation:**
The longitudinal trajectory of the "Elite" (universities that touched the Top 10) reveals a distinct pattern of stratification at the very top of the hierarchy.

1.  The **University of Oxford** (Light Blue line) demonstrates remarkable dominance, maintaining the #1 position consistently for nearly the entire decade. This suggests that the methodology heavily favors Oxford's specific mix of research volume and international outlook.

2.  A notable outlier is the **California Institute of Technology (Caltech)** (Dark Blue line). While it began the decade at #1 (2016), it has seen a gradual decline to #7 by 2026. Given the later findings on "Size Bias", this decline may be partly attributed to Caltech's extremely small student population (cf. https://registrar.caltech.edu/records/enrollment-statistics) compared to giants like Harvard or uToronto, making it vulnerable to volume-based metrics or reputation surveys.

3.  Perhaps the most significant finding is who is *missing*. While my earlier analysis showed a massive surge of Asian universities into the Top 200, **not a single Asian university** appears in this chart (Top 10). The absolute pinnacle of the THE ranking remains an exclusive club of US and UK institutions (plus ETH Zurich as the sole continental European representative). The "Glass Ceiling" for Asian universities appears to be right around Rank 11-15.


In [6]:
# Correlation Analysis (What matters most?)

# Let's look at the correlation between different scores and the Final Rank for 2026
score_cols = ['teaching', 'research_environment', 'research_quality', 'industry_impact', 'international_outlook', 'overall_score']
df_2026_corr = df_times[df_times['year'] == 2026][score_cols]
corr_matrix = df_2026_corr.corr()

# --- INTERACTIVE HEATMAP ---
fig = px.imshow(
    corr_matrix, 
    text_auto=".2f", 
    aspect="auto", 
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix of Ranking Indicators (2026)',
    template='plotly_white'
)
fig.show()

# --- GRAPH SUMMARY ---
print("\n--- GRAPH SUMMARY ---")
print("Factors most correlated with Overall Score (Sorted):")
print(corr_matrix['overall_score'].sort_values(ascending=False).to_string())


--- GRAPH SUMMARY ---
Factors most correlated with Overall Score (Sorted):
overall_score            1.000000
research_environment     0.909754
research_quality         0.873806
teaching                 0.820656
industry_impact          0.767323
international_outlook    0.657729



### Analysis of the Correlation Matrix

The heatmap above reveals the internal weighting mechanism of the *Times Higher Education* ranking system. By calculating the Pearson correlation coefficient between specific indicators and the final `overall_score`, I can determine which factors drive a university's success most heavily.

**Key Observations:**
* The strongest drivers of the final ranking are **Research Environment (0.91)** and **Research Quality (0.87)**. This confirms that the THE ranking is fundamentally a research-centric metric; universities cannot reach the top tier without massive research output and citation impact.
* The **Teaching** score also shows a very high correlation (**0.82**), suggesting that research-intensive universities typically maintain high staff-to-student ratios and doctoral output.
* Interestingly, **International Outlook** has the weakest correlation (**0.66**) with the overall score. This implies that a university can still achieve a very high global rank based on its research output alone, even if it is less "international" in its student/staff composition (I believe this is a trend often seen in elite US public universities).

**Conclusion:**
In contrast to the QS system, which heavily weighs reputation surveys, the THE system is statistically driven by hard research metrics.


In [7]:
# PRE-REQUISITE: Ensure Columns are ready for comparison

# 1. Clean QS Columns
# Define a mapping from the "Code Name" I want, to the "Actual Name" likely in the CSV
qs_map = {
    'ar_score': 'AR score', 
    'er_score': 'ER score', 
    'overall_score': 'Overall Score' # QS usually capitalizes this
}

for clean_name, potential_real_name in qs_map.items():
    # Check if the Capitalized version exists (most likely...)
    if potential_real_name in df_merged.columns:
        df_merged[f'{clean_name}_qs'] = pd.to_numeric(df_merged[potential_real_name], errors='coerce')
    # Fallback: Check if the lowercase version exists
    elif clean_name in df_merged.columns:
        df_merged[f'{clean_name}_qs'] = pd.to_numeric(df_merged[clean_name], errors='coerce')
    # Fallback: Check if it already has the suffix (e.g. overall_score_qs)
    elif f'{clean_name}_qs' in df_merged.columns:
        df_merged[f'{clean_name}_qs'] = pd.to_numeric(df_merged[f'{clean_name}_qs'], errors='coerce')

# 2. Clean THE Columns
# Since "overall_score" (THE) and "Overall Score" (QS) are different strings, 
# the merge might not have added suffixes automatically. I need to ensure they exist now.
the_cols = ['teaching', 'research_quality', 'industry_impact', 'overall_score']

for col in the_cols:
    # If the column exists without suffix (e.g. 'teaching'), create the '_times' version
    if col in df_merged.columns:
        df_merged[f'{col}_times'] = pd.to_numeric(df_merged[col], errors='coerce')
    # If it was already suffixed by the merge (e.g. 'teaching_times'), just ensure it's numeric
    elif f'{col}_times' in df_merged.columns:
        df_merged[f'{col}_times'] = pd.to_numeric(df_merged[f'{col}_times'], errors='coerce')

print("Score columns prepared for advanced analysis.")
# Print created columns to verify
print("Available QS columns:", [c for c in df_merged.columns if '_qs' in c])
print("Available Times columns:", [c for c in df_merged.columns if '_times' in c])

Score columns prepared for advanced analysis.
Available QS columns: ['name_clean_qs', 'overall_score_qs']
Available Times columns: ['name_clean_times', 'teaching_times', 'research_quality_times', 'industry_impact_times', 'overall_score_times']


In [8]:
# Insight 1: The "Prestige vs. Performance" Gap (Self-Healing Version)
# Hypothesis: QS favors Reputation, THE favors Metrics.

# --- Insight 1: The "Prestige vs. Performance" Gap (Self-Healing Version) ---

def get_actual_col_name(df, keywords):
    for col in df.columns:
        if all(k.lower() in col.lower() for k in keywords):
            return col
    return None

qs_col = get_actual_col_name(df_merged, ['ar', 'score']) 
if not qs_col: qs_col = get_actual_col_name(df_merged, ['academic', 'reputation'])

the_col = get_actual_col_name(df_merged, ['research', 'quality'])
if not the_col: the_col = get_actual_col_name(df_merged, ['research'])

print(f"Plotting using columns: QS='{qs_col}' vs THE='{the_col}'")

if qs_col and the_col:
    # --- INTERACTIVE VISUALIZATION ---
    fig = px.scatter(
        df_merged, 
        x=qs_col, 
        y=the_col, 
        color='region_group',
        hover_name='name_clean_times',  # <--- Shows name on hover!
        hover_data=['country'],
        title='Methodology Clash: QS Reputation vs. THE Research Quality',
        labels={qs_col: 'QS Academic Reputation', the_col: 'Times Research Quality'},
        template='plotly_white',
        width=900,
        height=600
    )
    
    # Add diagonal line
    fig.add_shape(type="line", x0=0, y0=0, x1=100, y1=100, line=dict(color="gray", dash="dash"))
    fig.show()
    
    print("\n--- GRAPH SUMMARY ---")
    metric_corr = df_merged[qs_col].corr(df_merged[the_col])
    print(f"Correlation between QS Reputation ({qs_col}) and THE Research ({the_col}): {metric_corr:.2f}")
else:
    print("Error: Could not find the necessary columns.")

Plotting using columns: QS='AR SCORE' vs THE='research_quality'



--- GRAPH SUMMARY ---
Correlation between QS Reputation (AR SCORE) and THE Research (research_quality): 0.45



### Analysis of Insight 1: The "Prestige vs. Performance" Gap

**Visual Observation:**
The scatter plot reveals a striking divergence between the two ranking methodologies, particularly in the top-left quadrant.

1.  **The "Hidden Gems" (Top-Left Quadrant):**
    * There is a dense cluster of universities with **high Research Quality** (Scores > 60 on the Times scale) but disproportionately **low Academic Reputation** (Scores < 40 on the QS scale).
    * **Regional Trend:** This area is heavily populated by **Asian and European institutions**. This suggests these universities are producing world-class research output *today*, but their global "brand recognition" has not yet caught up to their performance.

2.  **The "Old Guard" (Top-Right Diagonal):**
    * Universities along the diagonal line (where Reputation ≈ Research Quality) are predominantly from the **Anglosphere (USA/UK)**.
    * This indicates that for historic Western institutions, "Prestige" and "Performance" are closely aligned. Their reputation matches their output.

**Conclusion:**
This supports the hypothesis of a **Methodological Lag**. QS's reliance on reputation surveys acts as a "lagging indicator," favoring established brands. In contrast, THE's reliance on bibliometrics acts as a "leading indicator," rapidly identifying rising research powerhouses in Asia before they become household names globally.


In [9]:
# Insight 2: The Employability Paradox (Z-Score Version, cf: https://www.investopedia.com/terms/z/zscore.asp)
# Question: Do universities that help the industry (patents) get credit from employers (hiring)?
# Hypothesis: There is a gap between "Actual Industry Output" (THE) and "Employer Perception" (QS).

# --- Insight 2: The Employability Paradox (Z-Score Version) ---

col_employer_qs = get_actual_col_name(df_merged, ['er', 'score'])
if not col_employer_qs: col_employer_qs = get_actual_col_name(df_merged, ['employer', 'reputation'])

col_industry_the = get_actual_col_name(df_merged, ['industry', 'impact'])
if not col_industry_the: col_industry_the = get_actual_col_name(df_merged, ['industry'])

print(f"Analyzing Employability using: QS='{col_employer_qs}' vs THE='{col_industry_the}'")

if col_employer_qs and col_industry_the:
    df_merged['z_qs'] = zscore(df_merged[col_employer_qs], nan_policy='omit')
    df_merged['z_the'] = zscore(df_merged[col_industry_the], nan_policy='omit')
    df_merged['employability_gap'] = df_merged['z_qs'] - df_merged['z_the']

    top_brand_power = df_merged.sort_values('employability_gap', ascending=False).head(5)
    top_hidden_engines = df_merged.sort_values('employability_gap', ascending=True).head(5)

    # --- INTERACTIVE HISTOGRAM ---
    fig = px.histogram(
        df_merged, 
        x='employability_gap', 
        nbins=50,
        title='Distribution of the Employability Gap (Standardized Z-Scores)',
        labels={'employability_gap': 'Gap (Standard Deviations: Positive=Brand Dominant | Negative=Industry Dominant)'},
        template='plotly_white',
        color_discrete_sequence=['teal']
    )
    # Add vertical line at 0
    fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Balanced Profile")
    fig.show()

    print("--- GRAPH SUMMARY ---")
    print("\nTop 5 Positive Outliers (Brand Power - Z-Score Gap):")
    print(top_brand_power[['name_clean_times', 'employability_gap']].to_string(index=False))
    print("\nTop 5 Negative Outliers (Hidden Engines - Z-Score Gap):")
    print(top_hidden_engines[['name_clean_times', 'employability_gap']].to_string(index=False))

Analyzing Employability using: QS='overall_score' vs THE='industry_impact'


--- GRAPH SUMMARY ---

Top 5 Positive Outliers (Brand Power - Z-Score Gap):
                                name_clean_times  employability_gap
london school of economics and political science           2.406198
                              cornell university           2.347392
                           university of chicago           2.183149
                              harvard university           2.174177
                             columbia university           2.170297

Top 5 Negative Outliers (Hidden Engines - Z-Score Gap):
             name_clean_times  employability_gap
  national central university          -2.195523
         tokushima university          -2.006632
 yokohama national university          -1.881628
kyoto institute of technology          -1.763142
kyoto institute of technology          -1.763142


### Analysis of Insight 2: The "Utility vs. Prestige" Gap (Standardized)

To ensure a statistically robust comparison between the two differing scoring systems, this analysis uses **Z-Scores (Standard Standardization)** (cf. https://www.investopedia.com/terms/z/zscore.asp). It compared how many standard deviations a university performs above or below the global average.

**Observation 1: The "Old Money" Brand Power**
By standardizing the data, a clear pattern emerges at the top of the "Brand Gap" list. The institutions with the largest positive gap (Brand > Industry) are the **Global Elite of the Anglosphere** (e.g., *Harvard, Cornell, LSE, UChicago*).
* These universities possess such immense reputational capital that their "Brand Score" is often 2-3 standard deviations above the mean.
* Even if their industry output is excellent, their reputation is *statistically disproportionate*, confirming that for these historic institutions, "Prestige" is their most valuable asset.

**Observation 2:**
The list of negative outliers (Industry > Brand) identifies key technical powerhouses in East Asia (e.g., **National Central University** in Taiwan, **Tokushima University** in Japan).
* These universities show a Z-score gap of roughly -2.0, meaning their industrial performance is vastly superior to their global brand recognition.
* This confirms that the "Employability Paradox" is regional: Western universities benefit from a "Reputation Premium," while Asian technical universities suffer from a "Brand Deficit" despite world-class industrial output.

**Histogram Distribution:**
The Z-Score distribution follows a **Normal Curve (Bell Curve)** centered near zero.
* **Zero Line:** A score of 0 represents a "Balanced Profile," where a university's reputation closely matches its industrial output relative to the competition.
* **Symmetry:** The symmetry of the graph indicates that the "Employability Gap" is not a universal bias against all universities, but rather a polarizing force that separates "Old Guard" Prestige schools (Right Tail) from "New Economy" Technical schools (Left Tail).

In [10]:
# Insight 3: The Size Bias
# Hypothesis: Larger universities have an advantage in QS.

# --- Insight 3: The Size Bias ---

qs_rank_col = '2026_rank' if '2026_rank' in df_merged.columns else '2026 Rank'

if qs_rank_col in df_merged.columns:
    df_merged['rank_advantage_qs'] = pd.to_numeric(df_merged['rank'], errors='coerce') - pd.to_numeric(df_merged[qs_rank_col], errors='coerce')
    plot_data = df_merged.dropna(subset=['rank_advantage_qs', 'student_population'])

    # --- INTERACTIVE VISUALIZATION ---
    fig = px.scatter(
        plot_data, 
        x='student_population', 
        y='rank_advantage_qs',
        hover_name='name_clean_times', # <--- Name on hover!
        hover_data=['country', 'rank', qs_rank_col],
        title='Size Bias Analysis: Student Population vs. QS Rank Advantage',
        labels={'student_population': 'Student Population (Log Scale)', 'rank_advantage_qs': 'Rank Advantage in QS (vs Times)'},
        log_x=True, # Log Scale
        template='plotly_white',
        color_discrete_sequence=['teal']
    )
    fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Neutral Rank")
    fig.show()

    print("\n--- GRAPH SUMMARY ---")
    large_unis = plot_data[plot_data['student_population'] > 20000]
    small_unis = plot_data[plot_data['student_population'] <= 20000]
    print(f"Average QS Rank Advantage for Large Universities (>20k): {large_unis['rank_advantage_qs'].mean():.2f}")
    print(f"Average QS Rank Advantage for Small Universities (<=20k): {small_unis['rank_advantage_qs'].mean():.2f}")


--- GRAPH SUMMARY ---
Average QS Rank Advantage for Large Universities (>20k): 147.34
Average QS Rank Advantage for Small Universities (<=20k): 166.37


### Analysis of Insight 3: The "Size Bias"

**Understanding the Graph:**
* **Y-Axis:** Represents the "QS Advantage." A positive value means the university ranked **better in QS** than in Times Higher Education.
* **X-Axis:** Student Population (Log Scale).

**Observation:**
Contrary to the initial hypothesis, the data does **not** support a simple "Size Bias" favoring large institutions.
1.  **Small University Advantage:** The statistical summary reveals that smaller universities (<=20k students) actually hold a larger average rank advantage (**166 points**) compared to large universities (**147 points**).
2.  The scatter plot  shows a dense cluster of small/mid-sized institutions with high QS advantages. This suggests that while QS favors "Big Brand" reputation at the very top, it also likely benefits smaller, specialized institutions that get lost in the Times's volume-heavy research metrics.
   
**Conclusion:** The bias is likely not about *size* per se, but about *specialization*. QS appears more generous to smaller, focused institutions than the volume-centric Times methodology.

In [11]:
# Insight 3 & Augmentation: Economic Power vs. Academic Standing (with API)
# Try to use a Web API to fetch live economic data.
# Source: World Bank Open Data API (Indicator: NY.GDP.PCAP.CD = GDP per capita, current US$)

# --- Insight 3 & Augmentation: Economic Power vs. Academic Standing (with API) ---
import requests

url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=300&date=2023"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()[1]
    gdp_api_map = {entry['country']['value']: entry['value'] for entry in data if entry['value'] is not None}
else:
    gdp_api_map = {}

name_fixer = {
    'United States': 'United States', 'United Kingdom': 'United Kingdom', 'Korea, Rep.': 'South Korea',
    'Hong Kong SAR, China': 'Hong Kong', 'Russian Federation': 'Russia', 'Egypt, Arab Rep.': 'Egypt',
    'Iran, Islamic Rep.': 'Iran', 'Turkiye': 'Turkey', 'Singapore': 'Singapore', 'China': 'China',
    'Japan': 'Japan', 'Germany': 'Germany', 'Switzerland': 'Switzerland', 'Australia': 'Australia',
    'Canada': 'Canada', 'Netherlands': 'Netherlands', 'Sweden': 'Sweden'
}
for wb_name, times_name in name_fixer.items():
    if wb_name in gdp_api_map:
        gdp_api_map[times_name] = gdp_api_map[wb_name]

df_merged['country_gdp'] = df_merged['country'].map(gdp_api_map)
country_performance = df_merged.groupby('country')[['rank', 'country_gdp']].median().dropna().reset_index()

# --- PREPARE SPECIFIC LABELS ---
# Define which countries we want to always show on the screen
countries_to_label = [
    'United States', 'United Kingdom', 'China', 'Singapore', 'Switzerland', 
    'Germany', 'Australia', 'Canada', 'Japan', 'South Korea', 'Hong Kong', 
    'France', 'Netherlands', 'Sweden', 'India', 'Brazil', 'South Africa'
]

# Create a new column: If country is in our list, use its name. Otherwise, leave it blank ("")
country_performance['display_label'] = country_performance['country'].apply(
    lambda x: x if x in countries_to_label else ""
)

# --- INTERACTIVE VISUALIZATION ---
fig = px.scatter(
    country_performance, 
    x='country_gdp', 
    y='rank', 
    hover_name='country', 
    size='country_gdp',   
    text='display_label', # <--- THIS ADDS THE PERMANENT TEXT LABELS
    title='Economic Power (Live World Bank API Data) vs. Academic Standing',
    labels={'country_gdp': 'GDP Per Capita (USD - 2023)', 'rank': 'Median Times Rank (Lower is Better)'},
    template='plotly_white',
    color_discrete_sequence=['purple'],
    width=1000, # Made slightly wider to fit the text beautifully
    height=700
)

# Move the text slightly to the right of the bubble so it doesn't overlap
fig.update_traces(textposition='middle right', textfont=dict(size=11, color='black', weight='bold'))

fig.update_yaxes(autorange="reversed") # Rank 1 at top
fig.show()

print("\n--- GRAPH SUMMARY ---")
gdp_rank_corr = country_performance['country_gdp'].corr(country_performance['rank'])
print(f"Correlation between GDP per Capita and Median Rank: {gdp_rank_corr:.2f}")


--- GRAPH SUMMARY ---
Correlation between GDP per Capita and Median Rank: -0.70


### Analysis of Augmentation: Economic Power vs. Academic Standing

**Data Source:** Live data fetched via **World Bank Open Data API** (Indicator: *GDP per capita, current US$*, 2023).

**Observation 1:**
The scatter plot demonstrates a **strong negative correlation (-0.70)** between a nation's wealth (GDP per capita) and its median university ranking.
* *Note: A negative correlation here is a positive outcome, as higher wealth leads to a lower (better) rank number.*
* This confirms that academic excellence is capital-intensive. Countries like **Switzerland** and **Singapore** (top right of the graph) exemplify this, translating massive per-capita wealth directly into elite academic standing.

**Observation 2:**
**China** appears as a significant outlier above the trend line.
* Despite having a GDP per capita (~$13k) comparable to developing nations, its median university ranking rivals that of developed European economies.
* This suggests that strategic state planning (e.g., "Double First Class" initiative) can override pure economic determinism, allowing a nation to build world-class universities before becoming fully "wealthy."

**Observation 3:**
Interestingly, the absolute wealthiest nations per capita (**Luxembourg** and **Ireland**) do not hold the top academic spots.
* Their median ranks are relatively modest (~265 and ~361, respectively).
* This indicates that while money is necessary for academic prestige, it is not sufficient. A deep national research ecosystem (like that of the **US**, **UK**, or **Germany**) requires population scale and historical infrastructure that small, finance-hub nations may lack.


## General Conclusions

In this project, I analyzed the global higher education landscape to identify three key trends, while applying techniques such as data cleaning, merging, and API integration.

### 1. Longitudinal Trends (The "Eastward Shift")
Using 10 years of *Times Higher Education* data, it identified a structural shift in the global academic elite. The "Asian Tigers" (China, Singapore, Hong Kong) have more than doubled their presence in the Top 200 since 2016, eroding the historic dominance of the Anglosphere (US/UK).

### 2. Methodological Bias (Prestige vs. Performance)
By comparing the *QS 2026* and *Times 2026* datasets, the analysis uncovered significant methodological divergences:
* The analysis revealed that US and UK universities show high agreement between ranking systems. However, Asian and "Rest of World" institutions face a "Methodological Divergence," where their standing fluctuates significantly depending on whether the system prioritizes reputation (QS) or citations (Times).
* Using Z-Score standardization, the project identified that "Old Money" institutions (e.g., LSE, Harvard) rely heavily on Brand Reputation, whereas "Hidden Engines" in East Asia (e.g., Taiwan, Japan) produce world-class industrial output but suffer from a "Brand Deficit."
* Contrary to the initial hypothesis that "Bigger is Better," the data revealed that **smaller universities** actually held a slightly higher rank advantage in QS than larger ones. This suggests that QS's methodology may favor focused, specialized institutions just as much as large global brands, whereas the Times's metrics may penalize low-volume research output.

### 3. Socio-Economic Factors (API Augmentation)
By augmenting the dataset with live economic data from the **World Bank API**, it confirmed a general correlation between national wealth and academic performance. However, **China remains a massive outlier**, achieving elite rankings despite a significantly lower GDP per capita than its Western rivals. This suggests that strategic government policy, combined with a widespread societal emphasis on higher education and its prestige, can defy pure economic determinism.

However, these metrics should be interpreted with caution. As noted in a recent report by the *South China Morning Post* (2025, cf. [source](https://www.scmp.com/news/china/science/article/3332679/qs-university-rankings-ridiculed-china-latest-listing-bombs-online)), ranking outcomes can be volatile and disconnected from perceived academic reality. The report highlights that the *QS 2026* data faced significant backlash for "unintentionally funny" anomalies, such as ranking lesser-known Malaysian universities above research giants like the University of Tokyo. This suggests that in some cases, high rankings may reflect how well institutions tailor their strategies to specific metrics (such as "Internationalization" or reputational surveys) rather than indicating genuine academic superiority, supporting the view that commercial rankings can be "inevitably inflated."